# Dynamic Withdrawal Strategies - Step-by-Step Test

이 노트북은 Dynamic 전략 구현을 단계별로 테스트합니다.

## Step 1: 라이브러리 및 데이터 로드

In [22]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 로드 완료")

✅ 라이브러리 로드 완료


In [23]:
# 벤치마크 데이터 로드
with open('benchmark_data.pkl', 'rb') as f:
    data = pickle.load(f)

print(f"✅ 벤치마크 데이터 로드 완료")
print(f"   데이터 크기: {data.shape}")
print(f"   날짜 범위: {data.index[0].date()} ~ {data.index[-1].date()}")
print(f"\n컬럼:\n{data.columns.tolist()}")

✅ 벤치마크 데이터 로드 완료
   데이터 크기: (6521, 8)
   날짜 범위: 2001-01-03 ~ 2025-12-31

컬럼:
['미국성장주', '국내주식', '미국국채', '미국외국채', '신흥국달러채권', '국내중기채', '국내장기채', '금']


## Step 2: DataPreprocessor 테스트

In [24]:
from withdrawal_backtest import DataPreprocessor, PORTFOLIOS

print(f"DataPreprocessor 임포트 성공")
print(f"\n포트폴리오 목록:")
for i, port_name in enumerate(PORTFOLIOS.keys(), 1):
    port = PORTFOLIOS[port_name]
    print(f"  {i}. {port_name}: 목표 수익률={port['target_return']:.1f}%, 목표 변동성={port['target_risk']:.2f}%")

DataPreprocessor 임포트 성공

포트폴리오 목록:
  1. Port_4.0%: 목표 수익률=4.0%, 목표 변동성=3.75%
  2. Port_5.0%: 목표 수익률=5.0%, 목표 변동성=4.18%
  3. Port_6.0%: 목표 수익률=6.0%, 목표 변동성=5.00%
  4. Port_7.0%: 목표 수익률=7.0%, 목표 변동성=6.05%
  5. Port_8.0%: 목표 수익률=8.0%, 목표 변동성=7.18%
  6. Port_9.0%: 목표 수익률=9.0%, 목표 변동성=8.36%


In [25]:
# DataPreprocessor 실행
try:
    preprocessor = DataPreprocessor(data, add_portfolios=True)
    returns_df, month_starts = preprocessor.get_data()
    
    print(f"✅ DataPreprocessor 완료")
    print(f"   Returns DataFrame: {returns_df.shape}")
    print(f"   Month Starts Series: {month_starts.shape}")
    print(f"\n   Returns 컬럼:")
    print(f"   {returns_df.columns.tolist()[:10]}...")
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

=== 데이터 전처리 시작 ===
데이터 기간: 2001-01-03 ~ 2025-12-31
총 거래일: 6,521일
벤치마크: 8개

일별 수익률 계산 중... ✅ (6,520개 수익률)

수익률 통계 (연율화):
         연평균수익률   연변동성
미국성장주     12.16  19.90
국내주식      14.40  28.11
미국국채       4.18  11.06
미국외국채      3.34  11.36
신흥국달러채권    7.45  10.59
국내중기채      3.86   2.37
국내장기채      6.16   8.13
금         13.13  19.50

포트폴리오 수익률 계산:
  추가할 포트폴리오: 6개
  ✅ Port_4.0%: 연수익률 4.92%, 연변동성 2.65%
  ✅ Port_5.0%: 연수익률 6.01%, 연변동성 3.81%
  ✅ Port_6.0%: 연수익률 7.10%, 연변동성 5.39%
  ✅ Port_7.0%: 연수익률 8.28%, 연변동성 7.09%
  ✅ Port_8.0%: 연수익률 9.57%, 연변동성 8.95%
  ✅ Port_9.0%: 연수익률 10.87%, 연변동성 10.89%

월초 거래일 식별 중... ✅ (300개월)
첫 10개 월초: [datetime.date(2001, 1, 4), datetime.date(2001, 2, 1), datetime.date(2001, 3, 1), datetime.date(2001, 4, 2), datetime.date(2001, 5, 1), datetime.date(2001, 6, 1), datetime.date(2001, 7, 2), datetime.date(2001, 8, 1), datetime.date(2001, 9, 3), datetime.date(2001, 10, 1)]
✅ 전처리 완료

✅ DataPreprocessor 완료
   Returns DataFrame: (6520, 14)
   Month Starts Series: (6520,)

   Ret

## Step 3: Dynamic Simulator 테스트

In [26]:
from dynamic_simulator import GuardrailsWithdrawal, GuytonKlingerWithdrawal, DynamicWithdrawalSimulator

print("✅ Dynamic Simulator 클래스 임포트 성공")

# DynamicWithdrawalSimulator 생성
simulator = DynamicWithdrawalSimulator(returns_df, month_starts)
print(f"✅ DynamicWithdrawalSimulator 생성 완료")
print(f"   총 날짜: {len(simulator.dates)}")
print(f"   월초 개수: {np.sum(month_starts)}")

✅ Dynamic Simulator 클래스 임포트 성공
✅ DynamicWithdrawalSimulator 생성 완료
   총 날짜: 6520
   월초 개수: 300


## Step 4: Guardrails 백테스트 (간단한 테스트)

In [27]:
# 하나의 포트폴리오와 인출률로 빠른 테스트
test_portfolio = 'Port_5.0%'
test_wr = 0.05  # 5%
horizon_years = 10

print(f"테스트 설정:")
print(f"  포트폴리오: {test_portfolio}")
print(f"  인출률: {test_wr*100:.1f}%")
print(f"  기간: {horizon_years}년")

try:
    results_df = simulator.run_guardrails_backtest(
        benchmark=test_portfolio,
        horizon_years=horizon_years,
        initial_wr=test_wr,
        guardrail_width=0.20,
        inflation_rate=0.02,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ Guardrails 백테스트 완료")
    print(f"\n결과 DataFrame:")
    print(f"  크기: {results_df.shape}")
    print(f"  컬럼: {results_df.columns.tolist()}")
    print(f"\n샘플 데이터 (첫 5행):")
    display(results_df[['start_date', 'terminal_nav', 'total_withdrawal', 'is_fail']].head())
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

테스트 설정:
  포트폴리오: Port_5.0%
  인출률: 5.0%
  기간: 10년

Guardrails 백테스트: Port_5.0%, 10년, WR=5.0%
총 경로 수: 3,911


완료: 3,911개 경로

✅ Guardrails 백테스트 완료

결과 DataFrame:
  크기: (3911, 6)
  컬럼: ['start_date', 'terminal_nav', 'total_withdrawal', 'withdrawal_path', 'nav_path', 'is_fail']

샘플 데이터 (첫 5행):


,start_date,terminal_nav,total_withdrawal,is_fail
0,2001-01-04,115.287744,58.086801,False
1,2001-01-05,112.404324,56.163778,False
2,2001-01-08,112.404324,56.163778,False
3,2001-01-09,112.404324,56.163778,False
4,2001-01-10,112.404324,56.163778,False


## Step 5: 메트릭 계산 테스트

In [28]:
from metrics_calculator import MetricsCalculator

metrics_calc = MetricsCalculator()

try:
    metrics = metrics_calc.calculate_optimization_metrics(results_df, v0=100.0)
    
    print(f"✅ 메트릭 계산 완료")
    print(f"\n주요 메트릭:")
    print(f"  총 인출액 (평균): {metrics['total_withdrawal_mean']:,.2f}")
    print(f"  총 인출액 (범위): {metrics['total_withdrawal_worst']:,.2f} ~ {metrics['total_withdrawal_best']:,.2f}")
    print(f"  YoY 변동성 (평균): {metrics['yoy_volatility_mean']:.4f}")
    print(f"  YoY 변동성 (90분위): {metrics['yoy_volatility_90pct']:.4f}")
    print(f"  실패율: {metrics['failure_rate']:.2%}")
    print(f"  최종 NAV (중앙값): {metrics['terminal_nav_median']:.2%}")
    print(f"  경로 수: {metrics['total_paths']}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

✅ 메트릭 계산 완료

주요 메트릭:
  총 인출액 (평균): 57.16
  총 인출액 (범위): 52.48 ~ 61.51
  YoY 변동성 (평균): 0.0260
  YoY 변동성 (90분위): 0.0386
  실패율: 28.97%
  최종 NAV (중앙값): 10436.97%
  경로 수: 3911


## Step 6: Guyton-Klinger 백테스트

In [29]:
try:
    gk_results_df = simulator.run_guyton_klinger_backtest(
        benchmark=test_portfolio,
        horizon_years=horizon_years,
        initial_wr=test_wr,
        guardrail_width=0.20,
        adjustment_pct=0.10,
        freeze_threshold=-0.10,
        inflation_rate=0.02,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ Guyton-Klinger 백테스트 완료")
    print(f"\n결과 DataFrame:")
    print(f"  크기: {gk_results_df.shape}")
    print(f"\n샘플 데이터 (첫 5행):")
    display(gk_results_df[['start_date', 'terminal_nav', 'total_withdrawal', 'is_fail']].head())
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()


Guyton-Klinger 백테스트: Port_5.0%, 10년, WR=5.0%
총 경로 수: 3,911


완료: 3,911개 경로

✅ Guyton-Klinger 백테스트 완료

결과 DataFrame:
  크기: (3911, 6)

샘플 데이터 (첫 5행):


,start_date,terminal_nav,total_withdrawal,is_fail
0,2001-01-04,115.287744,58.086801,False
1,2001-01-05,112.404324,56.163778,False
2,2001-01-08,112.404324,56.163778,False
3,2001-01-09,112.404324,56.163778,False
4,2001-01-10,112.404324,56.163778,False


## Step 6.1: Guyton-Klinger 일별 Path 값 조회

`gk_results_df`의 `withdrawal_path`/`nav_path`는 월별 배열이므로, `get_single_path_detail`을 사용하여 일별 경로 데이터를 조회합니다.

In [ ]:
# ============================================================
# 시작일 설정 - 원하는 날짜로 변경하세요
# ============================================================
start_date = '2008-01-02'  # 원하는 시작일로 변경

# ============================================================
# 포트폴리오 상세 정보 출력
# ============================================================
print(f"\n{'='*60}")
print(f"포트폴리오 상세 정보")
print(f"{'='*60}")

portfolio_config = PORTFOLIOS.get(test_portfolio)
if portfolio_config:
    print(f"\n포트폴리오: {test_portfolio}")
    print(f"목표 수익률: {portfolio_config['target_return']:.2f}%")
    print(f"목표 변동성: {portfolio_config['target_risk']:.2f}%")
    print(f"\n자산 구성:")
    
    total_weight = 0.0
    for asset_kor, weight_pct in portfolio_config['weights'].items():
        print(f"  {asset_kor:15s}: {weight_pct:6.2f}%")
        total_weight += weight_pct
    
    print(f"  {'-'*30}")
    print(f"  {'합계':15s}: {total_weight:6.2f}%")
else:
    print(f"⚠️  포트폴리오 '{test_portfolio}' 설정을 찾을 수 없습니다.")

# ============================================================
# 일별 경로 조회 (get_single_path_detail)
# price_data를 전달하여 Price_* 컬럼 포함
# ============================================================
print(f"\n{'='*60}")
print(f"일별 경로 데이터 조회 (start_date: {start_date})")
print(f"{'='*60}")

daily_path_df = simulator.get_single_path_detail(
    portfolio=test_portfolio,
    start_date=start_date,
    strategy='guyton_klinger',
    horizon_years=horizon_years,
    initial_wr=test_wr,
    guardrail_width=0.20,
    adjustment_pct=0.10,
    freeze_threshold=-0.10,
    inflation_rate=0.02,
    v0=100.0,
    price_data=data
)

# ============================================================
# 일별 데이터 출력
# ============================================================
print(f"\n일별 경로 DataFrame:")
print(f"  크기: {daily_path_df.shape}")
print(f"  날짜 범위: {daily_path_df['Date'].iloc[0].date()} ~ {daily_path_df['Date'].iloc[-1].date()}")
print(f"  총 거래일: {len(daily_path_df)}")

# Price, Weight 컬럼 확인
price_cols = [c for c in daily_path_df.columns if c.startswith('Price_')]
weight_cols = [c for c in daily_path_df.columns if c.startswith('Weight_')]

print(f"\n컬럼 구성:")
print(f"  Price 컬럼: {len(price_cols)}개")
print(f"  Weight 컬럼: {len(weight_cols)}개")

# 주요 컬럼 표시
display_cols = ['Date', 'Total_NAV'] + price_cols + weight_cols + [
    'Monthly_Withdrawal', 'Is_Month_Start', 'Guardrail_Status'
]

print(f"\n--- 일별 데이터 (Total_NAV, Price Index, Weight) ---")
display(daily_path_df[display_cols])

In [39]:
daily_path_df.to_excel('guyton_klinger_path_details.xlsx', index=False)

## Step 6.5: Fixed Rate 백테스트

In [ ]:
try:
    fixed_results_df = simulator.run_fixed_backtest(
        benchmark=test_portfolio,
        horizon_years=horizon_years,
        initial_wr=test_wr,
        inflation_rate=0.02,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ Fixed Rate 백테스트 완료")
    print(f"\n결과 DataFrame:")
    print(f"  크기: {fixed_results_df.shape}")
    print(f"\n샘플 데이터 (첫 5행):")
    display(fixed_results_df[['start_date', 'terminal_nav', 'total_withdrawal', 'is_fail']].head())
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 6.6: Fixed Rate 일별 경로 조회

`fixed_results_df`의 `withdrawal_path`/`nav_path`는 월별 배열이므로, `get_single_path_detail`을 사용하여 일별 경로 데이터를 조회합니다.

In [ ]:
# ============================================================
# 시작일 설정 - 원하는 날짜로 변경하세요
# ============================================================
start_date = '2008-01-02'  # 원하는 시작일로 변경

# ============================================================
# 포트폴리오 상세 정보 출력
# ============================================================
print(f"\n{'='*60}")
print(f"포트폴리오 상세 정보")
print(f"{'='*60}")

portfolio_config = PORTFOLIOS.get(test_portfolio)
if portfolio_config:
    print(f"\n포트폴리오: {test_portfolio}")
    print(f"목표 수익률: {portfolio_config['target_return']:.2f}%")
    print(f"목표 변동성: {portfolio_config['target_risk']:.2f}%")
    print(f"\n자산 구성:")
    
    total_weight = 0.0
    for asset_kor, weight_pct in portfolio_config['weights'].items():
        print(f"  {asset_kor:15s}: {weight_pct:6.2f}%")
        total_weight += weight_pct
    
    print(f"  {'-'*30}")
    print(f"  {'합계':15s}: {total_weight:6.2f}%")
else:
    print(f"⚠️  포트폴리오 '{test_portfolio}' 설정을 찾을 수 없습니다.")

# ============================================================
# 일별 경로 조회 (get_single_path_detail)
# price_data를 전달하여 Price_* 컬럼 포함
# ============================================================
print(f"\n{'='*60}")
print(f"일별 경로 데이터 조회 (start_date: {start_date})")
print(f"{'='*60}")

fixed_daily_path_df = simulator.get_single_path_detail(
    portfolio=test_portfolio,
    start_date=start_date,
    strategy='fixed',
    horizon_years=horizon_years,
    initial_wr=test_wr,
    inflation_rate=0.02,
    v0=100.0,
    price_data=data
)

# ============================================================
# 일별 데이터 출력
# ============================================================
print(f"\n일별 경로 DataFrame:")
print(f"  크기: {fixed_daily_path_df.shape}")
print(f"  날짜 범위: {fixed_daily_path_df['Date'].iloc[0].date()} ~ {fixed_daily_path_df['Date'].iloc[-1].date()}")
print(f"  총 거래일: {len(fixed_daily_path_df)}")

# Price, Weight 컬럼 확인
price_cols = [c for c in fixed_daily_path_df.columns if c.startswith('Price_')]
weight_cols = [c for c in fixed_daily_path_df.columns if c.startswith('Weight_')]

print(f"\n컬럼 구성:")
print(f"  Price 컬럼: {len(price_cols)}개")
print(f"  Weight 컬럼: {len(weight_cols)}개")

# 주요 컬럼 표시
display_cols = ['Date', 'Total_NAV'] + price_cols + weight_cols + [
    'Monthly_Withdrawal', 'Is_Month_Start', 'Guardrail_Status'
]

print(f"\n--- 일별 데이터 (Total_NAV, Price Index, Weight) ---")
display(fixed_daily_path_df[display_cols].head())

# ============================================================
# 월초 인출액 확인 (인플레이션 조정 검증)
# ============================================================
print(f"\n{'='*60}")
print(f"월초 인출액 (인플레이션 조정 검증)")
print(f"{'='*60}")

month_start_withdrawals = fixed_daily_path_df[fixed_daily_path_df['Is_Month_Start'] == True][
    ['Date', 'Withdrawal_Amount', 'Year_Month']
].head(15)

print(f"\n처음 15개월 인출액:")
display(month_start_withdrawals)

## Step 7: WithdrawalOptimizer 테스트

In [ ]:
from optimizer import WithdrawalOptimizer

optimizer = WithdrawalOptimizer(returns_df, month_starts)

print(f"✅ WithdrawalOptimizer 생성 완료")

# 작은 범위로 빠른 테스트
withdrawal_rates = np.array([0.04, 0.05, 0.06])
constraints = {
    'max_yoy_volatility': 0.20,
    'max_failure_rate': 0.10,
    'min_terminal_nav': 0.40
}

print(f"\n테스트 설정:")
print(f"  인출률: {withdrawal_rates}")
print(f"  제약조건:")
print(f"    - Max YoY Volatility: {constraints['max_yoy_volatility']:.0%}")
print(f"    - Max Failure Rate: {constraints['max_failure_rate']:.0%}")
print(f"    - Min Terminal NAV: {constraints['min_terminal_nav']:.0%}")

In [ ]:
try:
    # 단일 포트폴리오 최적화 (빠른 테스트)
    portfolio_results = optimizer.optimize_single_portfolio(
        portfolio_name='Port_5.0%',
        withdrawal_rates=withdrawal_rates,
        horizon_years=10,
        constraints=constraints,
        v0=100.0,
        verbose=True
    )
    
    print(f"\n✅ 단일 포트폴리오 최적화 완료")
    print(f"\n결과 구조:")
    for wr, strategies in portfolio_results.items():
        print(f"  인출률 {wr:.2%}:")
        for strategy, metrics in strategies.items():
            feasible = "✓" if metrics.get('is_feasible') else "✗"
            total_w = metrics.get('total_withdrawal_mean', 0)
            print(f"    {strategy:20s} {feasible} - 총 인출액: {total_w:,.2f}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 8: 전체 최적화 (2개 포트폴리오)

In [ ]:
try:
    # 작은 범위로 빠른 테스트
    test_portfolios = ['Port_4.0%', 'Port_5.0%']
    test_wrs = np.array([0.04, 0.05, 0.06])
    
    results_df = optimizer.optimize_all_portfolios(
        portfolio_names=test_portfolios,
        withdrawal_rates=test_wrs,
        horizon_years=10,
        constraints=constraints,
        v0=100.0
    )
    
    print(f"\n✅ 전체 최적화 완료")
    print(f"\n결과 DataFrame: {results_df.shape}")
    print(f"\n전체 결과:")
    display(results_df.head(10))
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 9: 결과 요약

In [ ]:
try:
    print(f"\n최적화 결과 통계:")
    print(f"  전체 시나리오: {len(results_df)}")
    print(f"  가능한 시나리오 (제약 만족): {len(results_df[results_df['Feasible']==True])}")
    print(f"  불가능한 시나리오: {len(results_df[results_df['Feasible']==False])}")
    
    print(f"\n포트폴리오별 최적 솔루션:")
    for portfolio in results_df['Portfolio'].unique():
        df_port = results_df[results_df['Portfolio'] == portfolio]
        for strategy in ['fixed', 'guardrails', 'guyton_klinger']:
            df_strat = df_port[df_port['Strategy'] == strategy]
            df_feasible = df_strat[df_strat['Feasible'] == True]
            if not df_feasible.empty:
                best = df_feasible.loc[df_feasible['Total_Withdrawal'].idxmax()]
                print(f"  {portfolio:12s} {strategy:20s}: WR={best['WR']:.2%}, "
                      f"TotalW={best['Total_Withdrawal']:.0f}, "
                      f"Vol={best['YoY_Volatility']:.2%}")
            else:
                print(f"  {portfolio:12s} {strategy:20s}: No feasible solution")
                
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 10: UI 컴포넌트 테스트

In [ ]:
try:
    from dynamic_strategy_ui import (
        create_pareto_frontier_chart,
        create_strategy_comparison_chart
    )
    
    print(f"✅ UI 컴포넌트 임포트 성공")
    
    # Pareto Frontier 차트 생성
    portfolio_chart = 'Port_5.0%'
    fig = create_pareto_frontier_chart(results_df, portfolio_chart)
    
    print(f"✅ Pareto Frontier 차트 생성 완료")
    print(f"   차트 타입: {type(fig)}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
    import traceback
    traceback.print_exc()

## Step 11: Excel 검증용 데이터 내보내기

Step 6.1에서 생성한 `daily_path_df`에는 Price_* 및 Weight_* 컬럼이 포함되어 있어 Excel에서 수동 검증이 가능합니다.

In [ ]:
# Step 6.1에서 이미 daily_path_df가 생성되어 있으므로 바로 Excel로 내보내기
print("Excel 검증 방법:")
print("  Column A: Date")
print("  Column B: Total_NAV (Python 출력)")
print("  Column C~J: Price_* (8개 자산 가격 지수)")
print("  Column K~R: Weight_* (8개 자산 가중치, % 단위)")
print("  Column S: Portfolio_Return = SUMPRODUCT((Price[t]/Price[t-1]-1), Weight/100)")
print("  Column T: Total_NAV_Check = (이전NAV - 인출) × (1 + Portfolio_Return)")
print("  Column U: Diff = B - T → 0에 가까워야 함")